# Kazakh profanity detection — Colab notebook

Runs in a fresh Google Colab runtime. Two paths:

* **A. Use the trained model** (no training): install, load the model, type your own comment.
* **B. Reproduce everything**: build the dataset from KazSAnDRA, fine-tune GLiNER, evaluate on the test set.

For path B pick a GPU runtime: *Runtime → Change runtime type → T4 GPU* (≈15 min of training).

In [ ]:
# 1) Get the code
REPO_URL = "https://github.com/aizhankakharman/kazakh-profanity"   # <- set after pushing the repo
!git clone -q $REPO_URL kazakh-profanity
%cd kazakh-profanity

In [ ]:
# 2) Dependencies (Colab already has torch, transformers, pandas, sklearn, matplotlib)
!pip -q install "gliner[training]==0.2.30" datasketch==2.0.0 pymupdf python-docx
import torch, gliner, transformers
print("torch", torch.__version__, "| gliner", gliner.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())

## A. Predict with the trained model

The trained model is downloaded from the GitHub release **v1.1** (no training needed). The decision
threshold chosen on the validation set is stored next to the weights (`decision.json`) and loaded automatically.


In [ ]:
!wget -q -nc https://github.com/aizhankakharman/kazakh-profanity/releases/download/v1.1/kazakh-profanity-gliner.zip
!unzip -q -o kazakh-profanity-gliner.zip -d models/
MODEL = "models/kazakh-profanity-gliner"
from kzprof.inference import ProfanityDetector
det = ProfanityDetector(MODEL)
print("threshold:", det.threshold, "| device:", det.device)

In [ ]:
#@title Type your own Kazakh comment
text = "Телефон қотақ екен, екі аптада экраны сынды"  #@param {type:"string"}
res = det.predict(text)
print("label:", res["label"], "(1 = profane)  score:", res["score"])
print("profane words:", [(s["text"], round(s["score"], 3)) for s in res["spans"]])

In [ ]:
for t in ["Сапасы нашар, бір аптада сынып қалды.",
          "Курьер ақымақ екен, бірақ тауар жақсы",      # insult, not profanity -> 0
          "Көтергіш жүкті оңай көтереді",               # look-alike word -> 0
          "бл*ть тағы да жаңарту",                       # masked mat -> 1
          "Qotaq zat eken"]:                             # Latin transliteration -> 1
    r = det.predict(t); print(r["label"], f'{r["score"]:.3f}', t)

## B. Reproduce: data → training → evaluation

Each step writes its outputs into the repo folders (`data/processed/`, `models/`, `results/`).

In [ ]:
!python scripts/prepare_data.py          # downloads KazSAnDRA, labels, splits (seed 42)
!python scripts/augment_extra.py         # v1.1: phonetic + ALL-CAPS examples (train only)
!cat data/processed/stats.json | head -40

In [ ]:
# On a CUDA GPU the Apple-specific settings in configs/train.json are harmless.
!python scripts/train.py

In [ ]:
!python scripts/evaluate.py              # threshold on val, one evaluation on test
import json; m = json.load(open("results/metrics.json")); print(json.dumps(m["test"], indent=2))

In [ ]:
from IPython.display import Image, display
display(Image("results/confusion_matrix.png")); display(Image("results/pr_curve.png"))
import pandas as pd
pd.read_csv("results/errors.csv").head(15)

## Extras: redact a document, bleep a video

```bash
python scripts/redact_document.py examples/sample.pdf      # -> examples/sample.redacted.pdf
python scripts/censor_video.py examples/sample.mp4         # needs: pip install hydra-core omegaconf torchaudio
python app.py --share                                      # Gradio UI with all three tools
```